<a href="https://colab.research.google.com/github/rafaeldebarros/MVP_machine_learning/blob/main/MVP_ML_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 1. Carregar o arquivo JSON de metadados
json_url_metadados = "https://raw.githubusercontent.com/rafaeldebarros/MVP_machine_learning/refs/heads/main/metadados20260525.json"

metadados_25052026 = pd.read_json(json_url_metadados)
display(metadados_25052026.head())

# Converter para DataFrame do Pandas para facilitar a manipulação
df = pd.DataFrame(metadados_25052026)

# 2. Mapeamento de Desfecho Simplificado (Lógica Baseada na Aula de Classificação)
# Vamos criar uma função para rotular o desfecho com base no campo 'teor'
def mapear_desfecho(teor):
    if pd.isna(teor):
        return "Não Identificado"
    teor_lower = str(teor).lower()
    if "negando" in teor_lower or "não conhecendo" in teor_lower or "denegando" in teor_lower:
        return "Desfavorável"
    elif "provendo" in teor_lower or "concedendo" in teor_lower:
        return "Favorável"
    elif "parcial" in teor_lower:
        return "Parcialmente Favorável"
    return "Outros"

df['Resultado_Estimado'] = df['teor'].apply(mapear_desfecho)

# 3. Preparando o terreno para o NLP (Texto para análise)
# Como ainda não baixamos as íntegras em TXT, vamos usar a 'descricaoMonocratica' como nosso texto inicial
# Substituindo valores nulos por strings vazias
df['texto_analise'] = df['descricaoMonocratica'].fillna('')

# Criar uma coluna combinando o teor e o a descrição para dar mais corpo ao texto
df['texto_consolidado'] = df['teor'].fillna('') + " " + df['texto_analise']

print(f"Total de registros carregados: {len(df)}")
print(df[['processo', 'NM_MINISTRO', 'Resultado_Estimado', 'texto_consolidado']].head())

,situacao,dat_alteracao,codigo,descricao,cod_pai,cod_filhos,cod_filhos_ativos,nivel
0,A,NaN,13999,Multa do Artigo 467 da CLT,13970.0,NaN,NaN,5
1,A,NaN,14000,Multa do Artigo 477 da CLT,13970.0,NaN,NaN,5
2,A,NaN,14001,Saldo de Salário,13970.0,NaN,NaN,5
3,A,NaN,14002,Contrato de Experiência,13994.0,NaN,NaN,6
4,A,NaN,14003,Culpa Recíproca,13994.0,NaN,NaN,6


,SeqDocumento,dataPublicacao,tipoDocumento,numeroRegistro,processo,dataRecebimento,dataDistribuição,NM_MINISTRO,recurso,teor,descricaoMonocratica,assuntos
0,374496136,2026-05-25,DECISÃO,202504825421,AREsp 3128525,2025-12-04,2025-12-09,MARIA MARLUCE CALDAS,None,Não Conhecendo,Conheço do agravo de #{nome_da_parte} para não...,00287.03369.03370.
1,375924773,2026-05-25,ACÓRDÃO,202504918094,AREsp 3131300,2025-12-11,2025-12-18,CARLOS PIRES BRANDÃO,AgRg,Negando,None,"00287.05555., 01209.04305., 00287.03369.03372."
2,375955200,2026-05-25,DECISÃO,202600700129,AREsp 3188893,2026-02-27,2026-03-12,HUMBERTO MARTINS,None,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,00899.10431.10439.10441.
3,375924114,2026-05-25,ACÓRDÃO,202600769163,REsp 2261348,2026-03-04,2026-03-09,SEBASTIÃO REIS JÚNIOR,None,Negando,None,00287.03603.03607.
4,374149392,2026-05-25,DECISÃO,202600485736,AREsp 3178141,2026-02-12,2026-03-11,HERMAN BENJAMIN,EDcl,Negando,Embargos de Declaração de #{nome_da_parte} Não...,00014.05986.05990.14950.


Total de registros carregados: 3623
        processo            NM_MINISTRO Resultado_Estimado  \
0  AREsp 3128525   MARIA MARLUCE CALDAS       Desfavorável   
1  AREsp 3131300   CARLOS PIRES BRANDÃO       Desfavorável   
2  AREsp 3188893       HUMBERTO MARTINS       Desfavorável   
3   REsp 2261348  SEBASTIÃO REIS JÚNIOR       Desfavorável   
4  AREsp 3178141        HERMAN BENJAMIN       Desfavorável   

                                   texto_consolidado  
0  Não Conhecendo Conheço do agravo de #{nome_da_...  
1                                           Negando   
2  Não Conhecendo Não conhecido o recurso de #{no...  
3                                           Negando   
4  Negando Embargos de Declaração de #{nome_da_pa...  


In [11]:


# 2. Definição da composição da 1ª Seção do STJ (Direito Público)
# Nota: Incluímos os ministros da 1ª e 2ª Turmas que compõem a Seção.
ministros_1a_secao = [
    "BENEDITO GONÇALVES",
    "SERGIO KUKINA",
    "GURGEL DE FARIA",
    "FRANCISCO FALCÃO",
    "REGINA HELENA COSTA",
    "AFRÂNIO VILELA",
    "MARIA THEREZA DE ASSIS MOURA",
    "TEODORO SILVA SANTOS",
    "MARCO AURÉLIO BELLIZZE",
    "PAULO SÉRGIO DOMINGUES"
]


# 3. Aplicando o Filtro (Mapeamento de subconjunto de dados)
df_1a_secao = df[df['NM_MINISTRO'].isin(ministros_1a_secao)].copy()

# 4. Verificação dos Resultados
print(f"Total de decisões no dia: {len(df)}")
print(f"Total de decisões atribuídas à 1ª Seção: {len(df_1a_secao)}")
print("\nDistribuição de processos por Ministro na 1ª Seção neste dia:")
print(df_1a_secao['NM_MINISTRO'].value_counts())

Total de decisões no dia: 3623
Total de decisões atribuídas à 1ª Seção: 1324

Distribuição de processos por Ministro na 1ª Seção neste dia:
NM_MINISTRO
FRANCISCO FALCÃO                303
AFRÂNIO VILELA                  300
MARCO AURÉLIO BELLIZZE          261
MARIA THEREZA DE ASSIS MOURA    226
BENEDITO GONÇALVES               80
REGINA HELENA COSTA              76
PAULO SÉRGIO DOMINGUES           74
GURGEL DE FARIA                   3
TEODORO SILVA SANTOS              1
Name: count, dtype: int64


Agora vamos realizar o tratamento do atributo 'assunto', que aparece nos metadados apenas como códigos númericos.

O CNJ disponibiliza esses códigos e uma descrição destes na seguinte URL "https://dpj.cnj.jus.br/sgt/api/v1.0/assuntos.csv"

In [17]:
assuntos_url_csv = 'https://raw.githubusercontent.com/rafaeldebarros/MVP_machine_learning/refs/heads/main/assuntos.csv'

assuntos_cnj = pd.read_csv(assuntos_url_csv, sep = ";", encoding='utf-8')
display(assuntos_cnj.head())


assuntos_cnj.info()

,situacao,dat_alteracao,codigo,descricao,cod_pai,cod_filhos,cod_filhos_ativos,nivel
0,A,NaN,13999,Multa do Artigo 467 da CLT,13970.0,NaN,NaN,5
1,A,NaN,14000,Multa do Artigo 477 da CLT,13970.0,NaN,NaN,5
2,A,NaN,14001,Saldo de Salário,13970.0,NaN,NaN,5
3,A,NaN,14002,Contrato de Experiência,13994.0,NaN,NaN,6
4,A,NaN,14003,Culpa Recíproca,13994.0,NaN,NaN,6


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5601 entries, 0 to 5600
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   situacao           5601 non-null   object 
 1   dat_alteracao      1012 non-null   object 
 2   codigo             5601 non-null   int64  
 3   descricao          5601 non-null   object 
 4   cod_pai            5579 non-null   float64
 5   cod_filhos         852 non-null    object 
 6   cod_filhos_ativos  755 non-null    object 
 7   nivel              5601 non-null   int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 350.2+ KB


Verificando que os códigos foram interpetados como dados numéricos, precisamos transformá-los em strings para realizar futuras comparações e junções com os o conjunto de metadados de decisões judiciais.

Além disso, vou criar um dicionário de mapeamento para identificar de forma mais rápida os assuntos e sua descrição

In [19]:
# Garantir que a chave de cruzamento seja string e esteja limpa
assuntos_cnj['codigo_str'] = assuntos_cnj['codigo'].astype(str).str.strip()

# Criar um dicionário de mapeamento para busca ultrarrápida { 'codigo': 'descrição' }
mapa_assuntos = dict(zip(assuntos_cnj['codigo_str'], assuntos_cnj['descricao']))


Agora vou criar uma função que irá limpar os códigos do CNJ no* dataframe* de metadados, pois verifiquei que possuem uma construção diferente da utilizada pelo CNJ. Eles incluem zeros a esquerda para tornar todos os ccódigos do mesmo tamanho/extensão e vários códigos dos temas e subtemas estão separados apenas por ponto (.)

Por fim, a função vai relacionar os códigos com as descrições constantes no dicionários criado acima.



In [20]:
# 3. TRATANDO E TRADUZINDO OS CÓDIGOS DO STJ
def traduzir_codigos_cnj(string_assuntos, mapa):
    if pd.isna(string_assuntos) or string_assuntos.strip() == "":
        return []

    # Substituir vírgulas por pontos e quebrar a string por cada ponto '.'
    partes = string_assuntos.replace(',', '.').split('.')

    descricoes_traduzidas = []
    for parte in partes:
        parte_limpa = parte.strip()
        if parte_limpa:
            # Remove os zeros à esquerda (ex: "08826" vira "8826")
            codigo_limpo = str(int(parte_limpa))

            # Busca a tradução no dicionário de mapeamento de assuntos
            if codigo_limpo in mapa:
                descricoes_traduzidas.append(mapa[codigo_limpo])

    # Retorna uma lista de nomes sem duplicatas mantendo a ordem
    return list(dict.fromkeys(descricoes_traduzidas))

# Aplicando a função de tradução de NLP/Dados
df_1a_secao['assuntos_nomeados'] = df_1a_secao['assuntos'].apply(lambda x: traduzir_codigos_cnj(x, mapa_assuntos))



Processos da 1ª Seção mapeados com sucesso: 1324

Processo: AREsp 3220032 | Relator: BENEDITO GONÇALVES
Códigos Originais: 09985.09997.10022.10023., 09985.10157.10166.10168.
Assuntos Traduzidos: ['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO', 'Atos Administrativos', 'Infração Administrativa', 'Multas e demais Sanções', 'Organização Político-administrativa / Administração Pública', 'Conselhos Regionais de Fiscalização Profissional e Afins']
--------------------------------------------------
Processo: REsp 2263451 | Relator: REGINA HELENA COSTA
Códigos Originais: 09985.10219.10288.10305., 09985.10219.10288.10290.14716.
Assuntos Traduzidos: ['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO', 'Servidor Público Civil', 'Sistema Remuneratório e Benefícios', 'Gratificações de Atividade', 'Gratificação de Incentivo', 'Gratificação de Biênio (Adicional Bienal)']
--------------------------------------------------
Processo: AREsp 3221455 | Relator: BENEDITO GONÇALVES
C

In [21]:

# ==========================================
# 4. EXIBIÇÃO DOS RESULTADOS TRATADOS
# ==========================================
print(f"Processos da 1ª Seção mapeados com sucesso: {len(df_1a_secao)}\n")

# Amostra visual de como os dados ficaram estruturados
amostra = df_1a_secao[['processo', 'NM_MINISTRO', 'assuntos', 'assuntos_nomeados']].head(3)
for idx, row in amostra.iterrows():
    print(f"Processo: {row['processo']} | Relator: {row['NM_MINISTRO']}")
    print(f"Códigos Originais: {row['assuntos']}")
    print(f"Assuntos Traduzidos: {row['assuntos_nomeados']}\n" + "-"*50)

Processos da 1ª Seção mapeados com sucesso: 1324

Processo: AREsp 3220032 | Relator: BENEDITO GONÇALVES
Códigos Originais: 09985.09997.10022.10023., 09985.10157.10166.10168.
Assuntos Traduzidos: ['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO', 'Atos Administrativos', 'Infração Administrativa', 'Multas e demais Sanções', 'Organização Político-administrativa / Administração Pública', 'Conselhos Regionais de Fiscalização Profissional e Afins']
--------------------------------------------------
Processo: REsp 2263451 | Relator: REGINA HELENA COSTA
Códigos Originais: 09985.10219.10288.10305., 09985.10219.10288.10290.14716.
Assuntos Traduzidos: ['DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO', 'Servidor Público Civil', 'Sistema Remuneratório e Benefícios', 'Gratificações de Atividade', 'Gratificação de Incentivo', 'Gratificação de Biênio (Adicional Bienal)']
--------------------------------------------------
Processo: AREsp 3221455 | Relator: BENEDITO GONÇALVES
C

In [8]:
# Filtra apenas registros que possuem algum texto descritivo para não quebrar o modelo
df_com_texto = df[df['texto_consolidado'].str.strip() != ''].copy()

# 1. Vetorização com TF-IDF (Removendo palavras comuns em português)
vectorizer = TfidfVectorizer(stop_words=['de', 'a', 'o', 'que', 'do', 'da', 'para', 'em', 'com', 'no', 'na'])
X = vectorizer.fit_transform(df_com_texto['texto_consolidado'])

# 2. Clusterização com K-Means (Vamos tentar agrupar em 3 subtemas hipotéticos)
num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_com_texto['Subtema_Cluster'] = kmeans.fit_predict(X)

# Olhando o resultado dos grupos criados pela IA
for i in range(num_clusters):
    print(f"\n--- Amostra de Processos no Cluster {i} ---")
    print(df_com_texto[df_com_texto['Subtema_Cluster'] == i]['processo'].head(100).tolist())


--- Amostra de Processos no Cluster 0 ---
['AREsp 3128525', 'AREsp 3188893', 'AREsp 3201400', 'AREsp 3195281', 'AREsp 3220032', 'AREsp 3230479', 'AREsp 3221455', 'REsp 2259664', 'AREsp 3137846', 'AREsp 3137846', 'RHC    234261', 'AREsp 3158767', 'AREsp 3124799', 'AREsp 3111850', 'AREsp 3119816', 'AREsp 2848495', 'AREsp 3212253', 'HC 1097663', 'AREsp 3076461', 'HC 1048600', 'AREsp 3236302', 'EAREsp 2349841', 'AREsp 3029255', 'AREsp 3090023', 'AREsp 3148496', 'AREsp 3163834', 'CC 220597', 'AREsp 3223533', 'AREsp 3246061', 'AREsp 3241854', 'AREsp 3253016', 'REsp 2255209', 'AREsp 3228153', 'AREsp 3249381', 'HC 1096136', 'AREsp 3151213', 'AREsp 3185204', 'AREsp 3150702', 'AREsp 3166546', 'AREsp 3172092', 'AREsp 3138439', 'AREsp 3171136', 'REsp 2263326', 'AREsp 3026165', 'AREsp 3206931', 'AREsp 3222092', 'HC 1098860', 'AREsp 3250265', 'AREsp 3090993', 'AREsp 3246526', 'AREsp 3239250', 'HC 1098593', 'HC 1098694', 'AREsp 3095806', 'AREsp 3140031', 'AREsp 3142387', 'HC 1073855', 'HC 1068013', 